
## 📦 Install Dependencies

In [2]:
!pip -q install accelerate bitsandbytes datasets huggingface_hub peft scikit-learn transformers trl

## 📚 Libraries

In [9]:
from collections import Counter
from datasets import load_dataset, concatenate_datasets
from huggingface_hub import login, notebook_login
from google.colab import userdata
import os
from peft import LoraConfig
import numpy as np
import random
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer
import torch

## 🔐 Login to Hugging Face Hub

In [10]:
hf_token = userdata.get('HF_Token')

if hf_token:
    login(token=hf_token)
    print("HuggingFace login successful.")
else:
    print("HuggingFace token not found. Please set the HF_TOKEN environment variable or store it in Colab secrets.")



HuggingFace login successful.


## 📥 Load Medical Q and A Dataset

In [11]:
# Load Dataset
# https://huggingface.co/datasets/GBaker/MedQA-USMLE-4-options
dataset = load_dataset("GBaker/MedQA-USMLE-4-options")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

phrases_no_exclude_train.jsonl:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

phrases_no_exclude_test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1273 [00:00<?, ? examples/s]

In [12]:
# Inspect dataset
print(f"\ndataset['train'] head:")
print(dataset["train"][:5])

print("\ndataset['train'] column names:")
print(dataset["train"].column_names)


dataset['train'] head:
{'question': ['A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?', 'A 3-month-old baby died suddenly at night while asleep. His mother noticed that he had died only after she awoke in the morning. No cause of death was determined based on the autopsy. Which of the following precautions could have prevented the death of the baby?', "A mother brings her 3-week-old infant to the pediatrician's office because she is concerned about his feedin

🎲 Reproducibility

In [13]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


## 🧠 Define allowed answer letters and their token IDs

In [14]:
LETTER_CHOICES = ["A", "B", "C", "D"]

def row_to_prompt(example):
    """
    Convert a dataset row into:
      - prompt: chat-style instruction + question + options
      - target: the correct letter (A/B/C/D)
    """
    question = example["question"]
    options_dict = example["options"]  # dict like {'A': 'Ampicillin', ...}

    # Ensure options are in A/B/C/D order (some datasets are already)
    letters = sorted(options_dict.keys())

    options_text = "\n".join(f"{letter}. {options_dict[letter]}" for letter in letters)

    prompt = (
        "You are a medical expert answering USMLE-style questions.\n\n"
        f"Question:\n{question}\n\n"
        f"Options:\n{options_text}\n\n"
        "Answer with only the letter (A, B, C, or D)."
    )

    # Provided label is already like 'A', 'B', ...
    target = str(example["answer_idx"]).strip().upper()

    if target not in letters:
        raise ValueError(f"Target {target} not in options {letters}")

    return {"prompt": prompt, "target": target}

📌 Train/test split + mapping

In [15]:
# Original dataset has only 'train'. Split it.
split = dataset["train"].train_test_split(test_size=0.2, seed=SEED)

train_ds = split["train"].map(row_to_prompt)
test_ds  = split["test"].map(row_to_prompt)

print("\nSplit sizes:", len(train_ds), len(test_ds))
print("Train label distribution:", Counter(train_ds["target"]))
print("Test  label distribution:", Counter(test_ds["target"]))


Map:   0%|          | 0/8142 [00:00<?, ? examples/s]

Map:   0%|          | 0/2036 [00:00<?, ? examples/s]


Split sizes: 8142 2036
Train label distribution: Counter({'B': 2128, 'A': 2082, 'C': 2041, 'D': 1891})
Test  label distribution: Counter({'B': 526, 'C': 516, 'A': 502, 'D': 492})


⚖️ Balance the training  set (upsample minority classes)

In [17]:
def balance_dataset(ds, label_col="target", seed=42):
    """
    Upsample each class to match the largest class count.
    """
    counts = Counter(ds[label_col])
    max_count = max(counts.values())

    balanced_splits = []
    for label, count in counts.items():
        subset = ds.filter(lambda ex, lbl=label: ex[label_col] == lbl)

        repeat = max_count // count
        remainder = max_count % count

        pieces = [subset] * repeat
        if remainder > 0:
            extra = subset.shuffle(seed=seed).select(range(remainder))
            pieces.append(extra)

        balanced_splits.append(concatenate_datasets(pieces))

    balanced = concatenate_datasets(balanced_splits).shuffle(seed=seed)
    return balanced


train_ds_used = balance_dataset(train_ds, seed=SEED)
print("\n✅ Using balanced train set.")
print("Balanced distribution:", Counter(train_ds_used["target"]))


Filter:   0%|          | 0/8142 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8142 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8142 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8142 [00:00<?, ? examples/s]


✅ Using balanced train set.
Balanced distribution: Counter({'D': 2128, 'A': 2128, 'B': 2128, 'C': 2128})


# 🧠 Model

In [18]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

# Load tokenizer
tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=hf_token,
    use_fast=True,
    trust_remote_code=True,
)

# Ensure pad token exists
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# Baseline model in bf16 (not 4-bit)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
base_model.eval()

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

 🔢 Token IDs for choices

In [19]:
# Chat models often output " A" (leading space) as the next token.
# Score the tokens for " A", " B", " C", " D".
CHOICE_TEXTS = [" A", " B", " C", " D"]

choice_token_ids = []
for s in CHOICE_TEXTS:
    ids = tok(s, add_special_tokens=False).input_ids
    if len(ids) != 1:
        raise ValueError(
            f"Choice text {s!r} tokenized to {ids}. "
            "This script assumes single-token choices for logits scoring."
        )
    choice_token_ids.append(ids[0])

print("\nChoice token ids:", dict(zip(LETTER_CHOICES, choice_token_ids)))


Choice token ids: {'A': 362, 'B': 426, 'C': 356, 'D': 423}


🔮 Logits-based prediction helper

In [21]:
@torch.no_grad()
def predict_letter_from_logits(model, prompt: str) -> str:
    """
    Build a chat prompt, run model forward once, and choose A/B/C/D
    by comparing next-token logits for the tokens representing " A/B/C/D".
    """
    # Llama-instruct expects a chat template
    messages = [{"role": "user", "content": prompt}]
    prompt_text = tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tok(prompt_text, return_tensors="pt").to(model.device)
    outputs = model(**inputs)
    logits = outputs.logits  # [batch, seq_len, vocab]
    next_token_logits = logits[0, -1]  # [vocab]

    scores = [next_token_logits[tok_id].item() for tok_id in choice_token_ids]
    best = int(np.argmax(scores))
    return LETTER_CHOICES[best]

def evaluate_logits_model(model, ds, n=200, label="Model"):
    """
    Evaluate using the logits method on first n examples.
    """
    n = min(n, len(ds))
    y_true = [ds[i]["target"] for i in range(n)]
    y_pred = [predict_letter_from_logits(model, ds[i]["prompt"]) for i in range(n)]

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    print(f"\n{label} (N={n})")
    print(f"  Acc={acc:.3f}  Prec={prec:.3f}  Rec={rec:.3f}  F1={f1:.3f}\n")
    print("Classification report:")
    print(classification_report(y_true, y_pred, labels=LETTER_CHOICES, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=LETTER_CHOICES)
    print("Confusion matrix (rows=true, cols=pred):")
    print("    " + "  ".join(LETTER_CHOICES))
    for i, row in enumerate(cm):
        print(f" {LETTER_CHOICES[i]}  {row}")

    return y_true, y_pred, cm

📊 Baseline evaluation

In [22]:
 baseline_evaluation = evaluate_logits_model(base_model, test_ds, n=200, label="Baseline (bf16, no fine-tune)")


Baseline (bf16, no fine-tune) (N=200)
  Acc=0.615  Prec=0.660  Rec=0.619  F1=0.597

Classification report:
              precision    recall  f1-score   support

           A       0.80      0.31      0.44        52
           B       0.55      0.92      0.69        51
           C       0.72      0.56      0.63        52
           D       0.56      0.69      0.62        45

    accuracy                           0.61       200
   macro avg       0.66      0.62      0.60       200
weighted avg       0.66      0.61      0.60       200

Confusion matrix (rows=true, cols=pred):
    A  B  C  D
 A  [16 17  7 12]
 B  [ 0 47  0  4]
 C  [ 2 13 29  8]
 D  [ 2  8  4 31]


⚙️ Install + Reload Model in 4-bit (QLoRA-Ready)

In [25]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

qlora_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    device_map="auto",
    quantization_config=bnb_config, # Apply the 4-bit config
    trust_remote_code=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

🧩 Formatting function

In [28]:
def formatting_func(examples):
    """
    For SFTTrainer, create a single training string:
      <chat template for prompt> + " A/B/C/D"
    """
    texts = []
    for p, t in zip(examples["prompt"], examples["target"]):
        messages = [{"role": "user", "content": p}]
        prompt_text = tok.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        # Include space so it matches the scoring style
        texts.append(prompt_text + " " + t)
    return {"text": texts}

train_for_sft = train_ds_used.map(
    formatting_func,
    batched=True,
    remove_columns=train_ds_used.column_names,
)

eval_for_sft = test_ds.select(range(min(500, len(test_ds)))).map(
    formatting_func,
    batched=True,
    remove_columns=test_ds.column_names,
)

print("\nSFT example:\n", train_for_sft[0]["text"][:500], "...")

Map:   0%|          | 0/8512 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]


SFT example:
 <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

You are a medical expert answering USMLE-style questions.

Question:
A 40-year-old man comes to his doctor because of 2 weeks of progressively worsening pain on the outer side of his right elbow. He does not recall any trauma to the area. The patient plays tennis recreationally and has recently gone from playing weekly to  ...


🎯 Training Arguments

In [29]:
output_dir = "llama31-medqa-qlora"

args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    num_train_epochs=1,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=20,
    logging_first_step=True,
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    bf16=True,
    report_to="none",
    gradient_checkpointing=True,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


🚂 QLoRA SFTTrainer

In [30]:
trainer = SFTTrainer(
    model=qlora_model,
    peft_config=lora_config,
    train_dataset=train_for_sft,
    eval_dataset=eval_for_sft,
    args=args,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/8512 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8512 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8512 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

🚀 Fine-Tune Model with QLoRA

In [31]:

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss
1,2.358195
20,2.077471
40,1.227481
60,1.121003
80,1.095842
100,1.104839
120,1.065575
140,1.063169
160,1.053117
180,1.018526


TrainOutput(global_step=532, training_loss=1.0861498422192453, metrics={'train_runtime': 3775.6525, 'train_samples_per_second': 2.254, 'train_steps_per_second': 0.141, 'total_flos': 1.0344423736265933e+17, 'train_loss': 1.0861498422192453})

In [32]:
ft_model = trainer.model
ft_model.eval()


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Lin

Evaluate fine-tuned model (logits method)

In [33]:
ft_evaluation = evaluate_logits_model(ft_model, test_ds, n=200, label="Fine-tuned (QLoRA)")


Fine-tuned (QLoRA) (N=200)
  Acc=0.650  Prec=0.650  Rec=0.648  F1=0.647

Classification report:
              precision    recall  f1-score   support

           A       0.59      0.62      0.60        52
           B       0.68      0.78      0.73        51
           C       0.71      0.62      0.66        52
           D       0.62      0.58      0.60        45

    accuracy                           0.65       200
   macro avg       0.65      0.65      0.65       200
weighted avg       0.65      0.65      0.65       200

Confusion matrix (rows=true, cols=pred):
    A  B  C  D
 A  [32  8  7  5]
 B  [ 4 40  4  3]
 C  [ 7  5 32  8]
 D  [11  6  2 26]


Save adapter locally

In [34]:
adapter_dir = output_dir
ft_model.save_pretrained(adapter_dir)
tok.save_pretrained(adapter_dir)
print("\n✅ Saved adapter+tokenizer to:", adapter_dir)


✅ Saved adapter+tokenizer to: llama31-medqa-qlora
